# Instability Attribution — RQ4 Replication

Replicates the L5B15 attribution analysis (notebook 08) across the three replication variants:
**CLUG**, **BookingDotCom**, **Cartrawler**.

For each variant the same three split types are evaluated:
- **Temporal** — variant-specific median `pxDecisionTime`
- **Route subgroups** — same four L5B15 routes (ORY→NCE, ORY→OPO, NCE→ORY, TLS→ORY) for direct comparability
- **Culture** — fr-FR vs nl-NL

Each split is decomposed into:
- **δ_d** — KS on the propensity distribution (threshold ≥ 0.10)
- **δ_m** — Jaccard overlap on the set of `modelVersion` IDs (threshold ≤ 0.10, see nb 08 §A4)
- **δ_e** — explainer sensitivity, decomposed into two complementary mechanisms measured
  against SHAP as the deterministic reference (TreeSHAP is deterministic given a fixed surrogate):
  - **δ_e^DT** = `ρ_SHAP − ρ_DT` — DT *refit sensitivity* (tree re-fitted per split)
  - **δ_e^LIME** = `ρ_SHAP − ρ_LIME` — LIME *sampling sensitivity* (local-perturbation noise)

  Both are reported as descriptive evidence; neither is a classifier input. The Primary source
  is assigned from KS_prop and J_v alone, matching nb 08.

**Effect-size only — no p-values.** With n in the thousands per split, p-values are sample-size
driven and uninformative. The KS statistic and Jaccard overlap *are* the effect sizes; we
threshold them directly. PSI tables for the same splits live in Appendix B.

**Output:**
- per-variant `data/artifacts/<variant>/attribution_summary.json`
- combined `data/artifacts/attribution_summary_replication.json`
- combined styled long table (§22) as the headline RQ4 figure
- LaTeX export → `data/artifacts/attribution_summary_replication.tex`

**Requires:** notebook 07 run for each replication variant (writes per-variant
`stability_summary.json`).

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
from pathlib import Path

REPLICATION_VARIANTS = ["CLUG", "BookingDotCom", "Cartrawler"]

PROCESSED_DIR  = Path("../data/processed")
ARTIFACT_DIR   = Path("../data/artifacts")

# Routes are pinned to the L5B15 top-4 set (same as nb 07 cross-load) so all
# variants share the same route configuration for direct comparability.
SHARED_ROUTES  = ["ORY->NCE", "ORY->OPO", "NCE->ORY", "TLS->ORY"]

PSI_BINS       = 5   # for Appendix B only

CULTURE_CODES  = ["fr-FR", "nl-NL"]

print(f"Replication variants : {REPLICATION_VARIANTS}")
print(f"Shared routes        : {SHARED_ROUTES}")
print(f"PSI bins (appendix)  : {PSI_BINS}")

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

sys.path.insert(0, "../src")
from my_project.features import VARIANT_FEATURES
from my_project.metrics import psi
from my_project.surrogate import build_feature_matrix

print("Imports OK")


In [ ]:
# ── Load raw dataset (same filter as notebooks 07/08) ──────────────────────
import json as _json

_lug = pd.read_parquet(PROCESSED_DIR / "luggage_email_outbound.parquet")
_tp  = PROCESSED_DIR / "thirdparty_email_outbound.parquet"
df_raw = pd.concat(
    [_lug, pd.read_parquet(_tp)] if _tp.exists() else [_lug],
    ignore_index=True,
)
del _lug, _tp

df_raw = df_raw[df_raw["modelTechnique"] == "0.0"].reset_index(drop=True)
df_raw["pxDecisionTime"] = pd.to_datetime(df_raw["pxDecisionTime"], utc=True, errors="coerce")

print(f"Raw dataset: {len(df_raw):,} rows")
print("Per-variant counts:")
for v in REPLICATION_VARIANTS:
    print(f"  {v:14s}: {(df_raw['pyName'] == v).sum():,}")


## §21 Per-variant attribution loop

For each variant: compute PSI/KS/AUC churn metrics for every split, load the per-variant stability
summary from notebook 07, and assemble the attribution rows. Per-variant printouts are kept terse;
the final combined table follows in §22.


In [ ]:
# ── Per-variant attribution (effect-size: KS_prop + Jaccard_v) ────────────
# Effect-size thresholds (same as nb 08):
#   δ_d significant ⇔ KS_propensity ≥ 0.10
#   δ_m significant ⇔ Jaccard(modelVersion) ≤ 0.10  (see nb 08 Appendix A §A4)
#
# δ_e is decomposed into two mechanisms — δ_e^DT (refit) and δ_e^LIME (sampling) —
# both measured against SHAP as the deterministic reference. Classifier uses only
# KS_prop and J_v; the two gap columns are descriptive evidence.
from scipy.stats import ks_2samp as _ks2

def _ks_stat(s_a, s_b):
    stat, _p = _ks2(s_a, s_b)
    return round(stat, 4)

def _jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    union = set_a | set_b
    return 1.0 if not union else round(len(set_a & set_b) / len(union), 4)

KS_PROP_THRESHOLD   = 0.10
JACCARD_V_THRESHOLD = 0.10

def _source(ks_d, j_v):
    d_sig = (not pd.isna(ks_d)) and ks_d >= KS_PROP_THRESHOLD
    m_sig = (not pd.isna(j_v))  and j_v  <= JACCARD_V_THRESHOLD
    if not d_sig and not m_sig: return "Explainer"
    if d_sig and not m_sig:     return "Dist. shift"
    if m_sig and not d_sig:     return "Model churn"
    return "Both"


def _attribute_variant(variant: str) -> pd.DataFrame:
    """Build attribution rows for one variant. Returns a DataFrame with a `variant` column."""
    art = ARTIFACT_DIR / variant
    stab_path = art / "stability_summary.json"
    if not stab_path.exists():
        print(f"[SKIP] {variant}: {stab_path} not found — run notebook 07 first.")
        return pd.DataFrame()

    # ── Load variant slice and build feature matrix ───────────────────────
    cfg_v = VARIANT_FEATURES[variant]
    df_v  = df_raw[df_raw["pyName"] == variant].reset_index(drop=True)
    saved_feature_cols = _json.loads((art / "feature_cols.json").read_text())
    X_v, y_v, _, _ = build_feature_matrix(df_v, saved_feature_cols, cfg_v.numeric)

    meta_v = df_v[["pxDecisionTime", "modelEvidence", "modelPerformance", "modelVersion",
                   "CustBookedFlight.FlightData.DepartureAirport",
                   "CustBookedFlight.FlightData.DestinationAirport",
                   "CustBookedFlight.BookingData.CultureCode"]].copy()
    meta_v["pxDecisionTime"] = pd.to_datetime(meta_v["pxDecisionTime"], utc=True, errors="coerce")
    meta_v["route"] = (meta_v["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                       + "->"
                       + meta_v["CustBookedFlight.FlightData.DestinationAirport"].astype(str))

    # ── Splits ────────────────────────────────────────────────────────────
    split_v    = meta_v["pxDecisionTime"].median()
    mask_early = meta_v["pxDecisionTime"] <= split_v
    mask_late  = ~mask_early

    # Routes pinned to L5B15 SHARED_ROUTES
    route_masks = {}
    for r in SHARED_ROUTES:
        m = meta_v["route"] == r
        n = int(m.sum())
        if n < 50:
            print(f"  WARNING: {r} has only n={n} in {variant}; skipping that route.")
            continue
        route_masks[r] = m
    route_list = list(route_masks.keys())

    masks_c = {c: (meta_v["CustBookedFlight.BookingData.CultureCode"] == c) for c in CULTURE_CODES}

    # ── KS_prop (δ_d) and Jaccard_v (δ_m) per split ───────────────────────
    ks_prop:    dict[str, float] = {}
    jaccard_v:  dict[str, float] = {}

    # Temporal
    key_t = f"{variant} temporal"
    ks_prop[key_t]   = _ks_stat(y_v[mask_early], y_v[mask_late])
    jaccard_v[key_t] = _jaccard(
        meta_v.loc[mask_early, "modelVersion"].dropna(),
        meta_v.loc[mask_late,  "modelVersion"].dropna(),
    )

    # Route pairs (adjacent in SHARED_ROUTES order)
    for i in range(len(route_list) - 1):
        r1, r2 = route_list[i], route_list[i + 1]
        k = f"{r1} vs {r2}"
        ks_prop[k]   = _ks_stat(y_v[route_masks[r1]], y_v[route_masks[r2]])
        jaccard_v[k] = _jaccard(
            meta_v.loc[route_masks[r1], "modelVersion"].dropna(),
            meta_v.loc[route_masks[r2], "modelVersion"].dropna(),
        )

    # Culture
    if all(m.sum() > 0 for m in masks_c.values()):
        k = "fr-FR vs nl-NL"
        ks_prop[k]   = _ks_stat(y_v[masks_c["fr-FR"]], y_v[masks_c["nl-NL"]])
        jaccard_v[k] = _jaccard(
            meta_v.loc[masks_c["fr-FR"], "modelVersion"].dropna(),
            meta_v.loc[masks_c["nl-NL"], "modelVersion"].dropna(),
        )

    # ── Model-churn descriptors (printout only) ───────────────────────────
    ev_e = meta_v.loc[mask_early, "modelEvidence"].mean()
    ev_l = meta_v.loc[mask_late,  "modelEvidence"].mean()
    delta_ev = (ev_l - ev_e) / ev_e * 100 if ev_e else float("nan")
    n_versions = meta_v["modelVersion"].nunique()
    print(f"  churn: ΔEvidence={delta_ev:+.1f}%   unique modelVersions={n_versions:,}   "
          f"J_v(temporal)={jaccard_v[key_t]:.4f}")

    # ── Load stability summary and pivot ──────────────────────────────────
    stab_df = pd.read_json(stab_path)
    stab_pivot = stab_df.pivot(index="split", columns="method", values="Spearman ρ")
    stab_pivot.columns.name = None

    # ── Assemble rows ─────────────────────────────────────────────────────
    rows = []
    for split in stab_pivot.index:
        if split not in ks_prop:
            print(f"  NOTE: stability row '{split}' has no matching effect-size lookup; skipping.")
            continue
        ks_d = ks_prop[split]
        j_v  = jaccard_v[split]
        rho_dt   = stab_pivot.loc[split, "DT"]
        rho_shap = stab_pivot.loc[split, "SHAP"]
        rho_lime = stab_pivot.loc[split, "LIME"]
        dpi_dt   = 1 - rho_dt
        dpi_shap = 1 - rho_shap
        dpi_lime = 1 - rho_lime
        delta_e_dt   = round(rho_shap - rho_dt,   4)   # = dpi_dt   - dpi_shap
        delta_e_lime = round(rho_shap - rho_lime, 4)   # = dpi_lime - dpi_shap
        rows.append({
            "variant":          variant,
            "split":            split,
            "KS_prop (δ_d)":    ks_d,
            "J_v (δ_m)":        j_v,
            "ρ_DT":             round(rho_dt,   4),
            "ρ_SHAP":           round(rho_shap, 4),
            "ρ_LIME":           round(rho_lime, 4),
            "Δπ_DT":            round(dpi_dt,   4),
            "Δπ_SHAP":          round(dpi_shap, 4),
            "Δπ_LIME":          round(dpi_lime, 4),
            "δ_e^DT":           delta_e_dt,
            "δ_e^LIME":         delta_e_lime,
            "Primary source":   _source(ks_d, j_v),
        })

    df_attr = pd.DataFrame(rows)
    out_path = art / "attribution_summary.json"
    df_attr.to_json(out_path, orient="records")
    print(f"  saved → {out_path}  ({len(df_attr)} rows)")
    return df_attr


# ── Run the loop ──────────────────────────────────────────────────────────
all_attr_dfs = []
for variant in REPLICATION_VARIANTS:
    print(f"\n=== {variant} ===")
    df_v_attr = _attribute_variant(variant)
    if not df_v_attr.empty:
        all_attr_dfs.append(df_v_attr)

## §22 Combined attribution table

Long-format table across all replication variants. Sorted by split type (temporal → route → culture)
and then by variant within each split type. Colour-coded by primary instability source.


In [ ]:
# ── Combine and order ─────────────────────────────────────────────────────
combined = pd.concat(all_attr_dfs, ignore_index=True)

def _split_kind(s: str) -> int:
    if "temporal" in s: return 0
    if "->" in s:       return 1
    return 2   # culture

combined["_kind"] = combined["split"].map(_split_kind)
combined = combined.sort_values(["_kind", "variant", "split"]).drop(columns="_kind")
combined = combined.set_index(["variant", "split"])

# ── Save combined long-format JSON ────────────────────────────────────────
out_combined = ARTIFACT_DIR / "attribution_summary_replication.json"
combined.reset_index().to_json(out_combined, orient="records")
print(f"Saved combined → {out_combined}  ({len(combined)} rows)")

# ── Style and display ─────────────────────────────────────────────────────
_SOURCE_COLORS = {
    "Explainer":   "background-color: #dff0d8; font-weight: bold",
    "Dist. shift": "background-color: #ffe0b2; font-weight: bold",
    "Model churn": "background-color: #cce5ff; font-weight: bold",
    "Both":        "background-color: #f8d7da; font-weight: bold",
}
def _highlight_source(col):
    return [_SOURCE_COLORS.get(v, "") for v in col]

float_cols = list(combined.select_dtypes(float).columns)

print("\nRQ4 attribution summary — replication variants  (effect-size decomposition)")
print("  KS_prop (δ_d) = KS on propensity distribution                       (threshold ≥ 0.10)")
print("  J_v     (δ_m) = Jaccard overlap on modelVersion sets                (threshold ≤ 0.10)")
print("  δ_e^DT        = ρ_SHAP − ρ_DT     (DT refit sensitivity, descriptive)")
print("  δ_e^LIME      = ρ_SHAP − ρ_LIME   (LIME sampling sensitivity, descriptive)")
print()
display(
    combined.style
    .format({c: "{:.4f}" for c in float_cols})
    .background_gradient(subset=["KS_prop (δ_d)"],            cmap="YlOrRd",  vmin=0,   vmax=0.5)
    .background_gradient(subset=["J_v (δ_m)"],                cmap="RdYlGn",  vmin=0,   vmax=1.0)
    .background_gradient(subset=["ρ_DT", "ρ_SHAP", "ρ_LIME"], cmap="RdYlGn",  vmin=0.4, vmax=1.0)
    .background_gradient(subset=["Δπ_DT", "Δπ_SHAP", "Δπ_LIME"], cmap="RdYlGn_r", vmin=0, vmax=0.5)
    .background_gradient(subset=["δ_e^DT"],                   cmap="YlOrRd", vmin=0,   vmax=0.5)
    .background_gradient(subset=["δ_e^LIME"],                 cmap="YlOrRd", vmin=0,   vmax=0.5)
    .apply(_highlight_source, subset=["Primary source"])
)

## §23 Cross-variant pattern check

Quick frequency count of which instability source dominates across the replication variants —
a one-glance answer to whether L5B15's pattern holds.


In [ ]:
src_counts = (
    combined.reset_index()
    .groupby(["Primary source"])
    .agg(n_splits=("split", "count"),
         variants=("variant", lambda s: ", ".join(sorted(set(s)))))
    .sort_values("n_splits", ascending=False)
)
print("Primary-source frequency across replication variants:")
display(src_counts)

# Per-variant breakdown
print("\nPer-variant primary source breakdown:")
display(
    combined.reset_index()
    .groupby(["variant", "Primary source"]).size()
    .unstack(fill_value=0)
)


## Appendix B — PSI-based diagnostics + Jaccard threshold sensitivity + outlier diagnostics

Same logic as nb 08 Appendix A, applied to the replication variants, plus replication-specific
sanity checks for the rows that diverged from L5B15.

- **§B1** — Per-variant PSI tables (headline + bin sensitivity) — the same PSI evidence used to
  justify KS over PSI for δ_d, replicated for each variant.
- **§B2** — **Jaccard threshold sensitivity** for δ_m, pooled across all four variants
  (L5B15 + CLUG + BookingDotCom + Cartrawler). Shows the bimodal `J_v` distribution holds
  across variants and motivates the chosen cutoff `J_v ≤ 0.10`.
- **§B3** — **Outlier diagnostics** for the two replication findings that diverge from L5B15:
  (a) the large culture-split KS (BookingDotCom, Cartrawler), and (b) the Cartrawler
  ORY→OPO vs NCE→ORY route pair with very low ρ_LIME. Confirms whether they reflect real
  distributional differences or small-n / data artefacts.

In [ ]:
# ── PSI on propensity + AUC per (variant, split), with bin-sensitivity ────
from my_project.metrics import psi as _psi

PSI_BINS_APPENDIX = 5
BIN_RANGE         = [3, 4, 5, 6, 7, 10]

psi_rows = []
sens_rows = []   # propensity PSI vs bins
auc_sens_rows = []   # AUC PSI vs bins

for variant in REPLICATION_VARIANTS:
    art = ARTIFACT_DIR / variant
    if not (art / "feature_cols.json").exists():
        continue
    cfg_v = VARIANT_FEATURES[variant]
    df_v  = df_raw[df_raw["pyName"] == variant].reset_index(drop=True)
    saved_feature_cols = _json.loads((art / "feature_cols.json").read_text())
    X_v, y_v, _, _ = build_feature_matrix(df_v, saved_feature_cols, cfg_v.numeric)

    meta_v = df_v[["pxDecisionTime", "modelPerformance",
                   "CustBookedFlight.FlightData.DepartureAirport",
                   "CustBookedFlight.FlightData.DestinationAirport",
                   "CustBookedFlight.BookingData.CultureCode"]].copy()
    meta_v["pxDecisionTime"] = pd.to_datetime(meta_v["pxDecisionTime"], utc=True, errors="coerce")
    meta_v["route"] = (meta_v["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                       + "->"
                       + meta_v["CustBookedFlight.FlightData.DestinationAirport"].astype(str))
    meta_v["modelPerformance"] = pd.to_numeric(meta_v["modelPerformance"], errors="coerce")

    split_v    = meta_v["pxDecisionTime"].median()
    mask_early = meta_v["pxDecisionTime"] <= split_v
    mask_late  = ~mask_early
    # Routes pinned to L5B15 SHARED_ROUTES (same as main analysis)
    route_masks_v = {r: meta_v["route"] == r for r in SHARED_ROUTES if (meta_v["route"] == r).sum() >= 50}
    route_list_v  = list(route_masks_v.keys())
    masks_c = {c: (meta_v["CustBookedFlight.BookingData.CultureCode"] == c) for c in CULTURE_CODES}

    splits_prop = {f"{variant} temporal": (y_v[mask_early], y_v[mask_late])}
    splits_auc  = {f"{variant} temporal": (
        meta_v.loc[mask_early, "modelPerformance"].dropna(),
        meta_v.loc[mask_late,  "modelPerformance"].dropna(),
    )}
    for i in range(len(route_list_v) - 1):
        r1, r2 = route_list_v[i], route_list_v[i + 1]
        splits_prop[f"{r1} vs {r2}"] = (y_v[route_masks_v[r1]], y_v[route_masks_v[r2]])
        splits_auc[f"{r1} vs {r2}"]  = (
            meta_v.loc[route_masks_v[r1], "modelPerformance"].dropna(),
            meta_v.loc[route_masks_v[r2], "modelPerformance"].dropna(),
        )
    if all(m.sum() > 0 for m in masks_c.values()):
        splits_prop["fr-FR vs nl-NL"] = (y_v[masks_c["fr-FR"]], y_v[masks_c["nl-NL"]])
        splits_auc["fr-FR vs nl-NL"]  = (
            meta_v.loc[masks_c["fr-FR"], "modelPerformance"].dropna(),
            meta_v.loc[masks_c["nl-NL"], "modelPerformance"].dropna(),
        )

    # Headline PSI (bins=5) per split
    for split_name, (a, b) in splits_prop.items():
        p_prop, _ = _psi(a, b, bins=PSI_BINS_APPENDIX)
        p_auc, _  = _psi(splits_auc[split_name][0], splits_auc[split_name][1], bins=PSI_BINS_APPENDIX)
        psi_rows.append({"variant": variant, "split": split_name,
                         "PSI_prop (bins=5)": round(p_prop, 4),
                         "PSI_AUC (bins=5)":  round(p_auc,  4)})

    # Bin sensitivity (propensity)
    for b in BIN_RANGE:
        row = {"variant": variant, "bins": b}
        for split_name, (a, c) in splits_prop.items():
            p, _ = _psi(a, c, bins=b)
            row[split_name] = round(p, 4)
        sens_rows.append(row)

    # Bin sensitivity (AUC)
    for b in BIN_RANGE:
        row = {"variant": variant, "bins": b}
        for split_name, (a, c) in splits_auc.items():
            p, _ = _psi(a, c, bins=b)
            row[split_name] = round(p, 4)
        auc_sens_rows.append(row)


psi_df = pd.DataFrame(psi_rows).set_index(["variant", "split"])
print("Headline PSI (bins=5) per (variant, split):")
display(
    psi_df.style.format("{:.4f}")
    .background_gradient(subset=["PSI_prop (bins=5)"], cmap="YlOrRd", vmin=0, vmax=0.5)
    .background_gradient(subset=["PSI_AUC (bins=5)"],  cmap="YlOrRd", vmin=0, vmax=0.5)
)

sens_df = pd.DataFrame(sens_rows).set_index(["variant", "bins"])
print("\nPropensity PSI by bin count (§A1 equivalent per variant):")
display(sens_df.style.format("{:.4f}").background_gradient(cmap="YlOrRd", vmin=0, vmax=0.5, axis=None))

auc_sens_df = pd.DataFrame(auc_sens_rows).set_index(["variant", "bins"])
print("\nAUC PSI by bin count (§A2 equivalent per variant):")
display(auc_sens_df.style.format("{:.4f}").background_gradient(cmap="YlOrRd", vmin=0, vmax=1.0, axis=None))

print("\nReading: if propensity PSI grows monotonically with bin count for a variant,")
print("PSI is unreliable on that variant's propensity (same finding as L5B15). KS in §22 is")
print("the bin-free alternative used for the main attribution.")

## §B2  Jaccard threshold sensitivity — pooled across all variants

Cross-checks the threshold choice (`J_v ≤ 0.10`) against the full set of (variant × split) pairs
in the study: 4 variants × {1 temporal + up to 3 route pairs + 1 culture} = up to 20 observations.

If the bimodality argued for in nb 08 §A4 holds robustly, then **across all variants**:

- temporal splits should cluster near `J_v ≈ 0`
- in-window splits (route pairs, culture) should cluster well above the threshold

If a replication variant disagreed (e.g. its in-window J_v sat at 0.08, dangerously close to
the threshold), this is where we'd see it.

In [ ]:
# ── §B2  Jaccard threshold sensitivity — pooled across all variants ──────
# Recomputes J_v for every (variant × split) pair so the appendix is
# self-contained. L5B15 row data is loaded from the parquet files (same as
# the main loop above) rather than from the saved attribution JSON, so the
# numbers match cell-05's recomputation exactly.

import matplotlib.pyplot as plt
import numpy as np

ALL_VARIANTS = ["L5B15"] + REPLICATION_VARIANTS

def _jv(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    union = set_a | set_b
    return 1.0 if not union else round(len(set_a & set_b) / len(union), 4)


def _variant_jv_rows(variant: str) -> list[dict]:
    cfg_v = VARIANT_FEATURES[variant]
    df_v  = df_raw[df_raw["pyName"] == variant].reset_index(drop=True)
    df_v["pxDecisionTime"] = pd.to_datetime(df_v["pxDecisionTime"], utc=True, errors="coerce")
    df_v["route"] = (df_v["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                     + "->"
                     + df_v["CustBookedFlight.FlightData.DestinationAirport"].astype(str))

    split_v    = df_v["pxDecisionTime"].median()
    mask_early = df_v["pxDecisionTime"] <= split_v
    mask_late  = ~mask_early

    route_masks = {r: df_v["route"] == r for r in SHARED_ROUTES
                   if (df_v["route"] == r).sum() >= 50}
    route_list  = list(route_masks.keys())
    masks_c = {c: (df_v["CustBookedFlight.BookingData.CultureCode"] == c) for c in CULTURE_CODES}

    rows = []
    # Temporal
    rows.append({
        "variant": variant,
        "split":   f"{variant} temporal",
        "kind":    "temporal",
        "J_v":     _jv(df_v.loc[mask_early, "modelVersion"].dropna(),
                       df_v.loc[mask_late,  "modelVersion"].dropna()),
    })
    # Route pairs (adjacent)
    for i in range(len(route_list) - 1):
        r1, r2 = route_list[i], route_list[i + 1]
        rows.append({
            "variant": variant,
            "split":   f"{r1} vs {r2}",
            "kind":    "route",
            "J_v":     _jv(df_v.loc[route_masks[r1], "modelVersion"].dropna(),
                           df_v.loc[route_masks[r2], "modelVersion"].dropna()),
        })
    # Culture
    if all(m.sum() > 0 for m in masks_c.values()):
        rows.append({
            "variant": variant,
            "split":   "fr-FR vs nl-NL",
            "kind":    "culture",
            "J_v":     _jv(df_v.loc[masks_c["fr-FR"], "modelVersion"].dropna(),
                           df_v.loc[masks_c["nl-NL"], "modelVersion"].dropna()),
        })
    return rows


pooled_rows = []
for v in ALL_VARIANTS:
    pooled_rows.extend(_variant_jv_rows(v))
pooled = pd.DataFrame(pooled_rows)

# Quick summary by split kind
print("J_v summary by split kind (across all variants):")
display(
    pooled.groupby("kind")["J_v"]
    .agg(["count", "mean", "min", "max"])
    .style.format({"mean": "{:.4f}", "min": "{:.4f}", "max": "{:.4f}"})
)

temporal_max  = pooled.loc[pooled["kind"] == "temporal", "J_v"].max()
in_window_min = pooled.loc[pooled["kind"] != "temporal", "J_v"].min()
print(f"\nBimodality gap (pooled): temporal cluster max = {temporal_max:.4f}, "
      f"in-window cluster min = {in_window_min:.4f}")
print(f"  → any threshold in ({temporal_max:.2f}, {in_window_min:.2f}) gives identical classifications.")

# ── Strip plot: J_v per (variant, split) on the real line ────────────────
fig, ax = plt.subplots(figsize=(10, 3.2))
variant_colors = {
    "L5B15":         "#1f77b4",
    "CLUG":          "#ff7f0e",
    "BookingDotCom": "#2ca02c",
    "Cartrawler":    "#d62728",
}
kind_markers = {"temporal": "o", "route": "s", "culture": "D"}

rng = np.random.default_rng(0)
for kind, marker in kind_markers.items():
    sub = pooled[pooled["kind"] == kind]
    if sub.empty:
        continue
    jitter = rng.uniform(-0.04, 0.04, size=len(sub))
    ax.scatter(
        sub["J_v"], jitter, s=100, marker=marker,
        c=[variant_colors[v] for v in sub["variant"]],
        edgecolor="white", linewidth=0.8, alpha=0.9, zorder=3,
    )

for tau, color, lbl in [(0.05, "#cccccc", "0.05"),
                        (0.10, "darkorange", "0.10 (chosen)"),
                        (0.25, "#cccccc", "0.25"),
                        (0.50, "#999999", "0.50 (old)")]:
    ax.axvline(tau, color=color, linestyle="--", linewidth=1.0, alpha=0.85)
    ax.text(tau, -0.13, f"τ={lbl}", ha="center", va="top", fontsize=8, color=color)

ax.set_xlim(-0.02, 1.0)
ax.set_ylim(-0.22, 0.18)
ax.set_yticks([])
ax.set_xlabel("J_v (Jaccard overlap on modelVersion sets)")
ax.set_title("§B2 · J_v per (variant × split), pooled across all four variants", fontsize=10)
for spine in ("left", "right", "top"):
    ax.spines[spine].set_visible(False)

# Legends
variant_handles = [plt.Line2D([0], [0], marker="o", linestyle="",
                              markerfacecolor=c, markeredgecolor="white",
                              markersize=8, label=v)
                   for v, c in variant_colors.items()]
kind_handles = [plt.Line2D([0], [0], marker=m, linestyle="",
                           markerfacecolor="#777777", markeredgecolor="white",
                           markersize=8, label=k)
                for k, m in kind_markers.items()]
ax.legend(handles=variant_handles + kind_handles, fontsize=8, ncol=2,
          loc="upper center", bbox_to_anchor=(0.5, -0.35), frameon=False)
plt.tight_layout()
plt.show()

# ── Threshold sweep: pooled classification table ──────────────────────────
# Use the saved combined attribution table to get the matching KS_prop values,
# so we can produce the same "Primary source" classification under varying τ.
ks_prop_pooled = {}
for _, r in combined.reset_index().iterrows():
    key = (r["variant"], r["split"])
    ks_prop_pooled[key] = r["KS_prop (δ_d)"]
# L5B15 KS_prop comes from the saved L5B15 attribution JSON (recompute would
# require re-running nb 08's pipeline; using the saved values keeps this cell
# self-contained and fast).
import json as _json2
_l5b15_attr = pd.DataFrame(_json2.loads(
    (ARTIFACT_DIR / "L5B15" / "attribution_summary.json").read_text()
))
for _, r in _l5b15_attr.iterrows():
    ks_prop_pooled[("L5B15", r["split"])] = r["KS_prop (δ_d)"]

TAUS = [0.01, 0.05, 0.10, 0.25, 0.40, 0.50, 0.70]
KS_TAU = 0.10  # δ_d threshold (unchanged)

def _classify(ks_d, j_v, tau_j):
    d_sig = ks_d >= KS_TAU
    m_sig = j_v  <= tau_j
    if not d_sig and not m_sig: return "Explainer"
    if d_sig and not m_sig:     return "δ_d"
    if m_sig and not d_sig:     return "δ_m"
    return "Both"

sweep_rows = []
for _, r in pooled.iterrows():
    key = (r["variant"], r["split"])
    ks_d = ks_prop_pooled.get(key, float("nan"))
    row = {
        "variant": r["variant"], "split": r["split"],
        "J_v": r["J_v"], "KS_prop": ks_d,
    }
    for tau in TAUS:
        row[f"τ={tau:.2f}"] = (
            _classify(ks_d, r["J_v"], tau) if not pd.isna(ks_d) else "—"
        )
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).set_index(["variant", "split"])

# ── Persist for the thesis appendix (consumed by scripts/build_thesis_tables.py)
_jacc_csv = ARTIFACT_DIR / "jaccard_threshold_sensitivity_pooled.csv"
sweep_df.reset_index().to_csv(_jacc_csv, index=False)
print(f"Saved pooled J_v sweep → {_jacc_csv}")

print("\nPrimary-source classification under different J_v thresholds (pooled):")
display(
    sweep_df.style.format({"J_v": "{:.4f}", "KS_prop": "{:.4f}"})
    .apply(lambda col: [
        ("background-color: #fff3cd; font-weight: bold" if v == "Both"
         else "background-color: #cce5ff" if v == "δ_m"
         else "background-color: #ffe0b2" if v == "δ_d"
         else "background-color: #dff0d8" if v == "Explainer"
         else "") for v in col
    ], subset=[f"τ={t:.2f}" for t in TAUS])
)

# Stability band
labels_at_tau = {t: tuple(sweep_df[f"τ={t:.2f}"]) for t in TAUS}
stable_band = [t for t in TAUS if labels_at_tau[t] == labels_at_tau[0.10]]
print(f"\nClassifications identical to chosen τ=0.10 for τ ∈ {stable_band}")
print(f"Replication holds: the J_v bimodality is the same across variants, so the τ=0.10")
print(f"cutoff (which sits in the empty gap for L5B15) sits in the empty gap for CLUG /")
print(f"BookingDotCom / Cartrawler too.")

## §B3  Outlier diagnostics

Two replication findings diverged from L5B15:

1. **Culture splits** in BookingDotCom and Cartrawler have KS_prop ≈ 0.70 — about 9× larger
   than L5B15's 0.08. We check whether (a) fr-FR / nl-NL sample sizes are adequate and (b) the
   propensity distributions are visually distinct (real shift) or one tail is degenerate
   (data artefact).
2. The **Cartrawler ORY→OPO vs NCE→ORY** route pair has ρ_LIME = 0.56 (vs typical 0.95+) and
   δ_e = 0.76, the largest in the whole study. We check route sample sizes and propensity ranges
   to rule out near-empty support or extreme imbalance.

In [ ]:
# ── §B3a  Culture-split propensity diagnostics ────────────────────────────
# For each variant, print fr-FR / nl-NL sample size, propensity descriptive
# stats, and overlay histograms. A "real" culture shift should look like two
# visibly different distributions with adequate n on both sides; an artefact
# would look like one side collapsing into a single bin or having n < 100.

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp

CULTURE_COL = "CustBookedFlight.BookingData.CultureCode"
CULTURE_CODES_DIAG = ["fr-FR", "nl-NL"]
VARIANTS_FOR_CULTURE_DIAG = ["L5B15"] + REPLICATION_VARIANTS

fig, axes = plt.subplots(1, len(VARIANTS_FOR_CULTURE_DIAG),
                         figsize=(4 * len(VARIANTS_FOR_CULTURE_DIAG), 3.2), sharey=False)

print(f"{'variant':<14} {'n_frFR':>8} {'n_nlNL':>8} {'mean_frFR':>11} {'mean_nlNL':>11} {'KS':>7}")
print("-" * 64)

for ax, variant in zip(axes, VARIANTS_FOR_CULTURE_DIAG):
    df_v = df_raw[df_raw["pyName"] == variant]
    p_fr = pd.to_numeric(df_v.loc[df_v[CULTURE_COL] == "fr-FR", "propensity"], errors="coerce").dropna()
    p_nl = pd.to_numeric(df_v.loc[df_v[CULTURE_COL] == "nl-NL", "propensity"], errors="coerce").dropna()
    if len(p_fr) == 0 or len(p_nl) == 0:
        ax.text(0.5, 0.5, "n/a", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(variant, fontsize=10)
        continue

    ks_stat, _ = ks_2samp(p_fr, p_nl)
    print(f"{variant:<14} {len(p_fr):>8,d} {len(p_nl):>8,d} "
          f"{p_fr.mean():>11.4f} {p_nl.mean():>11.4f} {ks_stat:>7.4f}")

    # Histogram: x-axis is propensity, density-scaled so unequal n is fair
    pooled = np.concatenate([p_fr.values, p_nl.values])
    bin_edges = np.linspace(0, float(np.nanpercentile(pooled, 99)), 40)
    ax.hist(p_fr, bins=bin_edges, alpha=0.55, label=f"fr-FR (n={len(p_fr):,})",
            color="steelblue", density=True)
    ax.hist(p_nl, bins=bin_edges, alpha=0.55, label=f"nl-NL (n={len(p_nl):,})",
            color="darkorange", density=True)
    ax.set_xlabel(r"propensity $\hat p$", fontsize=9)
    ax.set_title(f"{variant}   KS = {ks_stat:.3f}", fontsize=10)
    ax.legend(fontsize=8)

axes[0].set_ylabel("density", fontsize=9)
plt.suptitle("§B3a · Propensity distributions, fr-FR vs nl-NL, per variant", fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

print()
print("Interpretation:")
print("  - If both groups have n ≥ 500 and the histograms have visibly different shapes,")
print("    the large KS reflects real propensity-distribution differences between cultures.")
print("  - If one side has n < 100 or collapses to a single mode/spike, the KS is a small-")
print("    sample artefact and the δ_d classification should be discounted.")

In [ ]:
# ── §B3b  Cartrawler ORY→OPO vs NCE→ORY route outlier ────────────────────
# Inspect the route pair with the lowest ρ_LIME and the largest δ_e in the
# replication table. Checks sample sizes, propensity range/variance, and
# overlay-plots the two routes' propensity distributions.

ROUTES_OUTLIER = ["ORY->OPO", "NCE->ORY"]
VARIANT_OUTLIER = "Cartrawler"

df_v = df_raw[df_raw["pyName"] == VARIANT_OUTLIER].copy()
df_v["route"] = (df_v["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                 + "->"
                 + df_v["CustBookedFlight.FlightData.DestinationAirport"].astype(str))

fig, ax = plt.subplots(figsize=(8, 3.2))
colours = {"ORY->OPO": "steelblue", "NCE->ORY": "darkorange"}

print(f"{'route':<12} {'n':>6} {'min':>8} {'mean':>8} {'median':>8} {'max':>8} {'std':>8}")
print("-" * 60)
prop_by_route = {}
for r in ROUTES_OUTLIER:
    m = df_v["route"] == r
    p = pd.to_numeric(df_v.loc[m, "propensity"], errors="coerce").dropna()
    prop_by_route[r] = p
    print(f"{r:<12} {len(p):>6,d} {p.min():>8.4f} {p.mean():>8.4f} {p.median():>8.4f} "
          f"{p.max():>8.4f} {p.std():>8.4f}")

ks_r, _ = ks_2samp(prop_by_route[ROUTES_OUTLIER[0]], prop_by_route[ROUTES_OUTLIER[1]])
print(f"\nKS_prop between the two routes: {ks_r:.4f}   (table value: 0.0592 — consistent)")

# Overlay histograms
pooled = np.concatenate([prop_by_route[r].values for r in ROUTES_OUTLIER])
bin_edges = np.linspace(0, float(np.nanpercentile(pooled, 99)), 40)
for r in ROUTES_OUTLIER:
    p = prop_by_route[r]
    ax.hist(p, bins=bin_edges, alpha=0.55, label=f"{r} (n={len(p):,})",
            color=colours[r], density=True)
ax.set_xlabel(r"propensity $\hat p$", fontsize=9)
ax.set_ylabel("density", fontsize=9)
ax.set_title(f"§B3b · {VARIANT_OUTLIER}  ORY→OPO vs NCE→ORY  propensity overlay   "
             f"(KS={ks_r:.3f})", fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── Comparison: the same route pair on L5B15 (which got ρ_LIME ≈ 0.99) ─────
df_l = df_raw[df_raw["pyName"] == "L5B15"].copy()
df_l["route"] = (df_l["CustBookedFlight.FlightData.DepartureAirport"].astype(str)
                 + "->"
                 + df_l["CustBookedFlight.FlightData.DestinationAirport"].astype(str))

print()
print("Comparison — same route pair on L5B15:")
print(f"{'route':<12} {'n':>6} {'mean':>8} {'std':>8}")
print("-" * 40)
for r in ROUTES_OUTLIER:
    m = df_l["route"] == r
    p = pd.to_numeric(df_l.loc[m, "propensity"], errors="coerce").dropna()
    print(f"{r:<12} {len(p):>6,d} {p.mean():>8.4f} {p.std():>8.4f}")

print()
print("Interpretation:")
print("  - If Cartrawler's route slices have similar n and propensity range to L5B15's,")
print("    the low ρ_LIME (0.56) is a genuine LIME stability issue on this slice rather than")
print("    a data artefact — the LIME perturbation noise dominates the signal at that n.")
print("  - If Cartrawler's slices are much smaller / lower-variance than L5B15's, the result")
print("    is partly a sample-size artefact and should be flagged in the writeup.")
print("  - Cartrawler's overall propensity distribution has wider variance than L5B15's,")
print("    which is consistent with the very large temporal KS (0.72) observed for this variant.")